In [1]:
import pandas as pd

In [2]:
bio = pd.read_csv("../data/raw/ppmi/Current_Biospecimen_Analysis_Results_28Jul2026.csv")

print(bio["TESTNAME"].dropna().unique())
print(len(bio["TESTNAME"].dropna().unique()))

C:\Users\eademola\AppData\Local\Temp\ipykernel_32520\2626008356.py:1: DtypeWarning: Columns (6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  bio = pd.read_csv("../data/raw/ppmi/Current_Biospecimen_Analysis_Results_28Jul2026.csv")


['ApoE Genotype' 'APOE GENOTYPE' 'DHPR' ... 'Diacetylspermidine'
 'Putrescine' 'Acetylputrescine']
8974


In [3]:
keywords = [
    "alpha",
    "synuclein",
    "aβ",
    "abeta",
    "amyloid",
    "tau",
    "nfl",
    "neurofilament",
]

for kw in keywords:
    matches = (
        bio.loc[
            bio["TESTNAME"].astype(str).str.contains(kw, case=False, na=False),
            "TESTNAME"
        ]
        .dropna()
        .unique()
    )

    print(f"\n--- {kw} ({len(matches)}) ---")
    print(matches)


--- alpha (5) ---
['CSF Alpha-synuclein' 'total alpha-Syn ELISA'
 'NEV alpha-synuclein (rep1)' 'NEV alpha-synuclein (rep2)' 'alpha-Syn-C']

--- synuclein (7) ---
['CSF Alpha-synuclein' 'a-Synuclein' 'NEV a-synuclein (rep1)'
 'NEV a-synuclein (rep2)' 'NEV a-synuclein' 'NEV alpha-synuclein (rep1)'
 'NEV alpha-synuclein (rep2)']

--- aβ (0) ---
[]

--- abeta (6) ---
['ABeta 1-42' 'ABeta' 'ABeta raw' 'ABeta42' 'ABeta40'
 'ABETA_DILUTION_FACTOR']

--- amyloid (0) ---
[]

--- tau (12) ---
['pTau' 'tTau' 'BD tTau' 'p217+tau' 'STAU1 (UniProt:O95793)' 'Ptau217p'
 'pTau181' 'eMTBR-TAU243' 'P_PTAU217' 'PTAU_DILUTION_FACTOR' 'PTAU217'
 'NPTAU217']

--- nfl (6) ---
['NfL' 'NFL' 'NFL (UniProt:P07196)' 'NFL_CORE (UniProt:P07196)'
 'NFL_CORE_pellet (UniProt:P07196)' 'NFL_pellet (UniProt:P07196)']

--- neurofilament (0) ---
[]


In [4]:
# SAA Biospecimen Analysis Results
saa = pd.read_csv("../data/raw/ppmi/SAA_Biospecimen_Analysis_Results_28Jul2026.csv")

print(saa["SAA_Status"].value_counts(dropna=False))
print(saa["SAAMethod"].value_counts(dropna=False))
print(saa["SAA_Type"].value_counts(dropna=False))

SAA_Status
Positive        3219
Negative        1735
Inconclusive     132
Name: count, dtype: int64
SAAMethod
Amprion-24h alpha-synuclein-SAA    3654
Amprion-Alpha-synuclein-SAA        1432
Name: count, dtype: int64
SAA_Type
NaN             2542
Type1           2349
Undetermined     150
Type2             45
Name: count, dtype: int64


### Exploring SAA file
There seems to be duplicates of the baseline visit

In [20]:
saa = pd.read_csv("../data/raw/ppmi/SAA_Biospecimen_Analysis_Results_28Jul2026.csv")
#saa = saa[["PATNO", "CLINICAL_EVENT", "SAA_Status"]]

cohort = pd.read_csv("../data/processed/cohort/ppmi_risk_cohort_multimodal_cdmgb.csv")



In [21]:
# Filter saa for participants in cohort
saa_cohort = saa[saa["PATNO"].isin(cohort["PATNO"])]

# Filter for baseline visits only
saa_cohort_bl = saa_cohort[saa_cohort["CLINICAL_EVENT"] == "BL"]

# Check for duplicates in PATNO
duplicates = saa_cohort_bl[saa_cohort_bl["PATNO"].duplicated(keep=False)]

# Count unique participants with duplicate BL visits
num_participants_with_duplicates = duplicates["PATNO"].nunique()

print(f"Total rows in saa_cohort_bl: {len(saa_cohort_bl)}")
print(f"Unique PATNO in saa_cohort_bl: {saa_cohort_bl['PATNO'].nunique()}")
print(f"Number of duplicate PATNO entries: {len(duplicates)}")
print(f"Number of participants with duplicate BL visits: {num_participants_with_duplicates}")

if len(duplicates) > 0:
    print("\nDuplicate PATNO entries:")
    print(duplicates[["PATNO", "CLINICAL_EVENT", "SAA_Status"]].sort_values("PATNO"))


# Interesting that I have 695 unique pateients here, the initilal number I had for the cdmbg group.

Total rows in saa_cohort_bl: 768
Unique PATNO in saa_cohort_bl: 695
Number of duplicate PATNO entries: 133
Number of participants with duplicate BL visits: 60

Duplicate PATNO entries:
       PATNO CLINICAL_EVENT    SAA_Status
1296    3264             BL      Positive
2710    3264             BL  Inconclusive
2711    3264             BL      Positive
909     3518             BL      Positive
1243    3518             BL  Inconclusive
...      ...            ...           ...
5032  166680             BL      Negative
354   182732             BL      Negative
520   182732             BL  Inconclusive
368   218096             BL      Positive
349   218096             BL      Positive

[133 rows x 3 columns]


In [22]:
# Check for exact duplicates (identical across all columns)
exact_duplicates = duplicates[duplicates.duplicated(subset=duplicates.columns.difference(['PATNO']), keep=False)]

print(f"Number of exact duplicate rows: {len(exact_duplicates)}")
print(f"Number of participants with exact duplicates: {exact_duplicates['PATNO'].nunique()}")

# Remove exact duplicates, keeping the first occurrence
saa_cohort_bl_dedup = saa_cohort_bl.drop_duplicates(subset=saa_cohort_bl.columns.difference(['PATNO']), keep='first')

# Recount duplicates after removing exact duplicates
duplicates_after = saa_cohort_bl_dedup[saa_cohort_bl_dedup['PATNO'].duplicated(keep=False)]

print(f"\nAfter removing exact duplicates:")
print(f"Total rows in saa_cohort_bl_dedup: {len(saa_cohort_bl_dedup)}")
print(f"Unique PATNO: {saa_cohort_bl_dedup['PATNO'].nunique()}")
print(f"Number of duplicate entries: {len(duplicates_after)}")
print(f"Number of participants with non-identical duplicate BL visits: {duplicates_after['PATNO'].nunique()}")

if len(duplicates_after) > 0:
    print("\nRemaining non-identical duplicate PATNO entries:")
    print(duplicates_after[["PATNO", "CLINICAL_EVENT", "SAA_Status", "RUNDATE"]].sort_values("PATNO"))

Number of exact duplicate rows: 0
Number of participants with exact duplicates: 0

After removing exact duplicates:
Total rows in saa_cohort_bl_dedup: 768
Unique PATNO: 695
Number of duplicate entries: 133
Number of participants with non-identical duplicate BL visits: 60

Remaining non-identical duplicate PATNO entries:
       PATNO CLINICAL_EVENT    SAA_Status     RUNDATE
1296    3264             BL      Positive  2023-01-04
2710    3264             BL  Inconclusive  2022-03-17
2711    3264             BL      Positive  2022-06-02
909     3518             BL      Positive  2023-01-20
1243    3518             BL  Inconclusive  2023-01-04
...      ...            ...           ...         ...
5032  166680             BL      Negative  2025-05-29
354   182732             BL      Negative  2023-10-04
520   182732             BL  Inconclusive  2023-09-29
368   218096             BL      Positive  2023-09-29
349   218096             BL      Positive  2023-10-04

[133 rows x 4 columns]


"218096","Male","Prodromal","BL","Cerebrospinal Fluid","Amprion-24h alpha-synuclein-SAA","Positive","Type1","139422","134634","157869","10.77","10.42","10.53","4.645E9","4.751E9","5.334E9","11.25","10.75","11.0","44.28","40.00","51.04","","","","","","","","","","","","","","","","2","2","2","","","NEV1","2023-10-04","237","Luis Concha","Amprion"

"218096","Male","Prodromal","BL","Cerebrospinal Fluid","Amprion-24h alpha-synuclein-SAA","Positive","Undetermined","147801","143254","37888","10.54","10.56","12.71","5.303E9","5.215E9","9.82152E8","11.01","11.01","16.02","45.78","44.98","2.93","","","","","","","","","","","","","","","","4","4","4","","","","2023-09-29","237","Luis Concha","Amprion"


In [19]:
saa_cohort_bl_dedup

,PATNO,CLINICAL_EVENT,SAA_Status
172,220392,BL,Positive
175,183051,BL,Negative
352,166680,BL,Inconclusive


In [16]:
# Check for duplicate PATNO in cohort
cohort_duplicates = cohort[cohort["PATNO"].duplicated(keep=False)]

print(f"Total rows in cohort: {len(cohort)}")
print(f"Unique PATNO in cohort: {cohort['PATNO'].nunique()}")
print(f"Number of duplicate PATNO entries: {len(cohort_duplicates)}")
print(f"Number of participants with duplicate entries: {cohort_duplicates['PATNO'].nunique()}")

if len(cohort_duplicates) > 0:
    print("\nDuplicate PATNO entries in cohort:")
    print(cohort_duplicates[["PATNO"]].sort_values("PATNO"))

Total rows in cohort: 752
Unique PATNO in cohort: 752
Number of duplicate PATNO entries: 0
Number of participants with duplicate entries: 0
